# Zededa HRIS — Rippling Data Cleaning
**Source:** Rippling HR Export (Excel)  
**Last Updated:** March 17, 2026  

Applies all cleaning steps and outputs a 25-column dataset to `zededa_hris_output.xlsx`.  
See `CLEANING_LOGIC.md` in the same folder for full documentation.

In [1]:
import pandas as pd
import numpy as np
from datetime import date
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

Libraries loaded.


In [2]:
# ── File Paths ───────────────────────────────────────────────────────────────
RIPPLING_FILE   = rf"160156_HR_Rippling_Report_-_for_Urja_3-2026.xlsx"
CONVERSION_FILE = rf"wise_fx_all_to_USD.xlsx"
OUTPUT_FILE     = rf"zededa_hris_output_mar27.xlsx"

# ── As-Of Date for ACTIVE employees only ─────────────────────────────────────
# !! UPDATE THIS DATE before each run (format: 'YYYY-MM-DD') !!
# Active employees     → this date is used for FX lookup
# Terminated employees → their Last work date is used automatically
AS_OF_DATE = '2026-03-27'

# ── Hard-coded EOR Rippling profile numbers ──────────────────────────────────
EOR_PROFILE_NUMBERS = {'235', '277', '284', '291'}

# ── Employment types that get annualized when base pay < 500 ─────────────────
ANNUALIZE_TYPES = {'Hourly, part-time', 'Temporary / Intern'}

print("Configuration loaded.")
print(f"  Rippling file   : {RIPPLING_FILE}")
print(f"  Conversion file : {CONVERSION_FILE}")
print(f"  Output file     : {OUTPUT_FILE}")
print(f"  AS_OF_DATE      : {AS_OF_DATE}  ← active employees; termed use Last work date")

Configuration loaded.
  Rippling file   : 160156_HR_Rippling_Report_-_for_Urja_3-2026.xlsx
  Conversion file : wise_fx_all_to_USD.xlsx
  Output file     : zededa_hris_output_mar27.xlsx
  AS_OF_DATE      : 2026-03-27  ← active employees; termed use Last work date


In [3]:
# ── Load Rippling HR data ─────────────────────────────────────────────────────
df = pd.read_excel(RIPPLING_FILE)

# ── Load FX conversion rates ──────────────────────────────────────────────────
fx_df = pd.read_excel(CONVERSION_FILE, dtype={'concat': str})
fx_df['concat'] = fx_df['concat'].str.strip()  # keep as string for exact match

print(f"Rippling data   : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Conversion data : {fx_df.shape[0]} rows × {fx_df.shape[1]} columns")
print()
print("Rippling columns:")
for c in df.columns:
    print(f"  {c!r}")
print()
print("Conversion sample:")
print(fx_df.head(3).to_string())

Rippling data   : 308 rows × 18 columns
Conversion data : 35975 rows × 5 columns

Rippling columns:
  'Employee'
  'Rippling profile number'
  'Title'
  'Employment type name'
  'Start date'
  'Last work date'
  'Department'
  'Work location'
  'Base compensation - Currency'
  'Base compensation'
  'On-target commission - Currency'
  'On-target commission'
  'Target annual bonus percent'
  'Grade Level'
  'Operating Title'
  'Work location - City'
  'Work location - State'
  'Work location - Country'

Conversion sample:
        date source_currency target_currency    concat  exchange_rate
0  8/11/2015             AED             USD  42227AED       0.272257
1  8/12/2015             AED             USD  42228AED       0.272257
2  8/13/2015             AED             USD  42229AED       0.272257


In [4]:
# ── Step 1: Remove Contractors ────────────────────────────────────────────────
before = len(df)
df = df[df['Employment type name'].fillna('').str.strip() != 'Contractor'].copy()
removed = before - len(df)
print(f"Step 1 ✓  Removed {removed} contractor row(s). Remaining: {len(df)}")

Step 1 ✓  Removed 60 contractor row(s). Remaining: 248


In [5]:
# ── Step 2: Remove rows with blank Rippling profile number ───────────────────
before = len(df)
df = df[df['Rippling profile number'].notna()].copy()
df = df[df['Rippling profile number'].astype(str).str.strip() != ''].copy()
removed = before - len(df)
print(f"Step 2 ✓  Removed {removed} row(s) with blank Rippling profile number. Remaining: {len(df)}")

Step 2 ✓  Removed 16 row(s) with blank Rippling profile number. Remaining: 232


In [6]:
# ── Step 3: Fill blanks in Department and Work location ───────────────────────
for col in ['Department', 'Work location']:
    df[col] = df[col].fillna('Unclassified')
    df[col] = df[col].replace('', 'Unclassified')
    count = (df[col] == 'Unclassified').sum()
    print(f"Step 3 ✓  '{col}': {count} value(s) set to 'Unclassified'")

Step 3 ✓  'Department': 2 value(s) set to 'Unclassified'
Step 3 ✓  'Work location': 0 value(s) set to 'Unclassified'


In [7]:
# ── Step 4: Clean Work Location State ────────────────────────────────────────
def clean_work_state(row):
    country = str(row.get('Work location - Country') or '').strip()
    state   = str(row.get('Work location - State')   or '').strip()
    if country == 'United States':
        return state
    if country == 'India' and state == 'Karnataka':
        return state
    return ''

df['Work location - State'] = df.apply(clean_work_state, axis=1)
blanked = (df['Work location - State'] == '').sum()
print(f"Step 4 ✓  Work location - State: {blanked} value(s) blanked out (non-US / non-Karnataka India)")

Step 4 ✓  Work location - State: 75 value(s) blanked out (non-US / non-Karnataka India)


In [8]:
# ── Step 5: Normalize Grade Level (strip trailing .0) ────────────────────────
def normalize_grade(val):
    if pd.isna(val) or str(val).strip() == '':
        return val
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return val

df['Grade Level'] = df['Grade Level'].apply(normalize_grade)
sample = df['Grade Level'].dropna().unique()[:10].tolist()
print(f"Step 5 ✓  Grade Level normalized. Sample values: {sample}")

Step 5 ✓  Grade Level normalized. Sample values: [8.0, 7.0, 10.0, 9.0, 11.0, 6.0, 12.0, 5.0, 13.0, 14.0]


In [9]:
# ── Step 6a: Add as_of_date column ───────────────────────────────────────────
# Terminated (Last work date is filled) → use Last work date
# Active     (Last work date is blank)  → use AS_OF_DATE

from datetime import date as date_type
_excel_epoch  = date_type(1899, 12, 30)
_AS_OF_SERIAL = (pd.Timestamp(AS_OF_DATE).date() - _excel_epoch).days  # kept for diagnostic cells

def get_as_of_date(last_work_date):
    if pd.notna(last_work_date) and str(last_work_date).strip() not in ('', 'NaT', 'nan'):
        return pd.Timestamp(last_work_date).date()
    return pd.Timestamp(AS_OF_DATE).date()

df['as_of_date'] = df['Last work date'].apply(get_as_of_date)

active_count = (df['as_of_date'] == pd.Timestamp(AS_OF_DATE).date()).sum()
termed_count = (df['as_of_date'] != pd.Timestamp(AS_OF_DATE).date()).sum()
print(f"as_of_date assigned:")
print(f"  Active     → {AS_OF_DATE}        : {active_count} employees")
print(f"  Terminated → own Last work date   : {termed_count} employees")
print()

# ── Step 6b: Pull exchange rate from FX file for each employee ────────────────
# Key format: str(Excel serial of as_of_date) + currency code  e.g. "46097EUR"
# Full precision rate — no rounding

def make_concat_key(currency, as_of_date=None):
    d = as_of_date or pd.Timestamp(AS_OF_DATE).date()
    return str((d - _excel_epoch).days) + str(currency).strip().upper()

def get_fx_rate(currency, as_of_date):
    """Return full-precision exchange rate from FX file. USD = 1.0. Not found = None."""
    if pd.isna(currency) or str(currency).strip() == '':
        return None
    currency = str(currency).strip().upper()
    if currency == 'USD':
        return 1.0
    key   = make_concat_key(currency, as_of_date)
    match = fx_df[fx_df['concat'] == key]
    if match.empty:
        print(f"  ⚠ No rate found: {currency} on {as_of_date}  (key: {key})")
        return None
    return float(match.iloc[0]['exchange_rate'])  # full precision

df['_base_rate'] = df.apply(
    lambda r: get_fx_rate(r['Base compensation - Currency'], r['as_of_date']), axis=1)
df['_comm_rate'] = df.apply(
    lambda r: get_fx_rate(r['On-target commission - Currency'], r['as_of_date']), axis=1)

print(f"Exchange rates pulled (full precision).")
print(f"  Base rate not found : {df['_base_rate'].isna().sum()} employees")
print(f"  Comm rate not found : {df['_comm_rate'].isna().sum()} employees")

# Sample — non-USD employees with their as_of_date and rate
_sample = df[df['Base compensation - Currency'].fillna('').str.upper() != 'USD'][
    ['Rippling profile number', 'Last work date', 'as_of_date',
     'Base compensation - Currency', '_base_rate']].head(8)
print("\nSample non-USD employees:")
print(_sample.to_string(index=False))

as_of_date assigned:
  Active     → 2026-03-27        : 120 employees
  Terminated → own Last work date   : 112 employees

Exchange rates pulled (full precision).
  Base rate not found : 0 employees
  Comm rate not found : 0 employees

Sample non-USD employees:
 Rippling profile number Last work date as_of_date Base compensation - Currency  _base_rate
                   130.0     08/12/2023 2023-08-12                          EUR    1.094600
                   158.0     11/30/2023 2023-11-30                          EUR    1.097350
                   173.0     12/29/2023 2023-12-29                          INR    0.012026
                   244.0            NaN 2026-03-27                          EUR    1.153450
                   137.0            NaN 2026-03-27                          EUR    1.153450
                    13.0            NaN 2026-03-27                          EUR    1.153450
                   152.0            NaN 2026-03-27                          EUR    1.153450
  

In [10]:
# ── Step 6c: Convert non-USD amounts to USD ───────────────────────────────────
# USD      → use as-is, no change
# Non-USD  → amount × rate
# No rate  → 'Not Available'

def convert_to_usd(amount, currency, rate):
    if pd.isna(amount):
        return np.nan
    amount   = float(amount)
    currency = str(currency).strip().upper() if pd.notna(currency) else ''
    if currency in ('USD', ''):
        return amount           # USD or blank currency → no conversion
    if rate is None:
        return 'Not Available'
    return round(amount * rate, 4)

df['_Annual Base Pay'] = df.apply(
    lambda r: convert_to_usd(
        r['Base compensation'],
        r['Base compensation - Currency'],
        r['_base_rate']), axis=1)

df['_Annualized Sales Comp'] = df.apply(
    lambda r: convert_to_usd(
        r['On-target commission'],
        r['On-target commission - Currency'],
        r['_comm_rate']), axis=1)

print("Step 6c ✓  Non-USD → USD conversion done.")
print(f"  Annual Base Pay       — 'Not Available' : {(df['_Annual Base Pay'] == 'Not Available').sum()}")
print(f"  Annualized Sales Comp — 'Not Available' : {(df['_Annualized Sales Comp'] == 'Not Available').sum()}")

Step 6c ✓  Non-USD → USD conversion done.
  Annual Base Pay       — 'Not Available' : 0
  Annualized Sales Comp — 'Not Available' : 0


In [11]:
# ── Step 6d: Annualize Hourly / Temp employees ───────────────────────────────
# Only applies to: 'Hourly, part-time'  and  'Temporary / Intern'
# Check is on the RAW original amount (before USD conversion)
# If raw amount < 500  →  multiply converted USD amount × 30 × 52

_ann_types = df['Employment type name'].isin(ANNUALIZE_TYPES)

# Annual Base Pay annualization — check raw Base compensation < 500
_raw_base   = pd.to_numeric(df['Base compensation'], errors='coerce')
_apply_base = _ann_types & (_raw_base < 500)
_base_num   = pd.to_numeric(df['_Annual Base Pay'], errors='coerce')
df.loc[_apply_base, '_Annual Base Pay'] = (_base_num[_apply_base] * 30 * 52).round(4)

print(f"Step 6d ✓  Annual Base Pay annualized {_apply_base.sum()} employee(s) (raw amount < 500, × 30 × 52):")
if _apply_base.sum() > 0:
    _show = df[_apply_base][['Rippling profile number', 'Employment type name',
                              'Base compensation', 'Base compensation - Currency',
                              '_base_rate', '_Annual Base Pay']]
    print(_show.to_string(index=False))

# Annualized Sales Comp annualization — check raw On-target commission < 500
_raw_comm   = pd.to_numeric(df['On-target commission'], errors='coerce')
_apply_comm = _ann_types & (_raw_comm < 500)
_comm_num   = pd.to_numeric(df['_Annualized Sales Comp'], errors='coerce')
df.loc[_apply_comm, '_Annualized Sales Comp'] = (_comm_num[_apply_comm] * 30 * 52).round(4)

print(f"\nStep 6d ✓  Annualized Sales Comp annualized {_apply_comm.sum()} employee(s) (raw commission < 500, × 30 × 52):")
if _apply_comm.sum() > 0:
    _show2 = df[_apply_comm][['Rippling profile number', 'Employment type name',
                               'On-target commission', 'On-target commission - Currency',
                               '_comm_rate', '_Annualized Sales Comp']]
    print(_show2.to_string(index=False))

# ── Step 6e: Zero Annual Base Pay → 0.1 ──────────────────────────────────────
_base_final = pd.to_numeric(df['_Annual Base Pay'], errors='coerce')
_zero_mask  = _base_final == 0
df.loc[_zero_mask, '_Annual Base Pay'] = 0.1
print(f"\nStep 6e ✓  Annual Base Pay zeros replaced with 0.1 : {_zero_mask.sum()} employee(s)")

# ── Step 6f: Bonus percent — use as-is, no formula ───────────────────────────
df['_Bonus Pct'] = pd.to_numeric(df['Target annual bonus percent'], errors='coerce').round(4)
print(f"Step 6f ✓  Bonus % taken as-is from source.")

Step 6d ✓  Annual Base Pay annualized 8 employee(s) (raw amount < 500, × 30 × 52):
 Rippling profile number Employment type name  Base compensation Base compensation - Currency  _base_rate  _Annual Base Pay
                   113.0   Temporary / Intern               18.0                          USD    1.000000           28080.0
                   166.0    Hourly, part-time               35.0                          USD    1.000000           54600.0
                   105.0   Temporary / Intern               18.0                          USD    1.000000           28080.0
                   214.0    Hourly, part-time               65.0                          USD    1.000000          101400.0
                   176.0   Temporary / Intern               72.0                          USD    1.000000          112320.0
                    34.0   Temporary / Intern                0.0                          INR    0.013518               0.0
                    11.0   Temporary / Intern    

In [12]:
# ── Step 7: Assign Base Category Name ────────────────────────────────────────
def assign_base_category(rippling_id):
    # Strip .0 that pandas adds when reading numeric IDs from Excel
    emp_id_str = str(rippling_id).strip().split('.')[0]
    if emp_id_str in EOR_PROFILE_NUMBERS:
        return '60010 - Salary and Wages - EOR'
    return '60020 - Salary and Wages'

df['_Base Category Name'] = df['Rippling profile number'].apply(assign_base_category)
eor_count = (df['_Base Category Name'] == '60010 - Salary and Wages - EOR').sum()
print(f"Step 7 ✓  Base Category assigned.")
print(f"  EOR employees  (60010): {eor_count}")
print(f"  Standard       (60020): {len(df) - eor_count}")

Step 7 ✓  Base Category assigned.
  EOR employees  (60010): 4
  Standard       (60020): 228


In [13]:
# ── Build Output DataFrame (25 columns) ──────────────────────────────────────

def fmt_date(val):
    """Format a date value as mm/dd/yyyy string; return blank string if null."""
    if pd.isna(val) or str(val).strip() in ('', 'NaT', 'nan'):
        return ''
    try:
        return pd.Timestamp(val).strftime('%m/%d/%Y')
    except Exception:
        return ''

def clean_emp_id(val):
    """Convert Rippling ID to clean string (no .0 from float)."""
    if pd.isna(val):
        return ''
    return str(val).strip().split('.')[0]


output_df = pd.DataFrame({
    'Emp ID'               : df['Rippling profile number'].apply(clean_emp_id),
    'Req ID'               : '',
    'Req opening ID'       : '',
    'Name (masked)'        : df['Employee'],
    'Job Title'            : df['Title'],
    'Start Date'           : df['Start date'].apply(fmt_date),
    'Termination Date'     : df['Last work date'].apply(fmt_date),
    'Department'           : df['Department'],
    'Location'             : df['Work location'],
    'Base Category Name'   : df['_Base Category Name'],
    'Bonus Category Name'  : '60030 - Bonus',
    'Annual Base Pay'      : df['_Annual Base Pay'],
    'Bonus %'              : df['_Bonus Pct'],
    'Annualized Stock Comp': '',
    'Annualized Sales Comp': df['_Annualized Sales Comp'],
    'Grade'                : df['Grade Level'],
    'Supervisor'           : '',
    'Work City'            : '',
    'Work State'           : df['Work location - State'],
    'Work Country'         : df['Work location - Country'],
    'Effective Start Date' : '',
    'Effective End Date'   : '',
    'Employee Type'        : df['Employment type name'],
    'Hire Type'            : '',
    'Termination Type'     : '',
})

print(f"Output DataFrame built: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
output_df.head(5)

Output DataFrame built: 232 rows × 25 columns


,Emp ID,Req ID,Req opening ID,Name (masked),Job Title,Start Date,Termination Date,Department,Location,Base Category Name,...,Grade,Supervisor,Work City,Work State,Work Country,Effective Start Date,Effective End Date,Employee Type,Hire Type,Termination Type
0,130,,,Marcus Oliver Menzel,"Director of Sales, EMEA",11/01/2022,08/12/2023,Sales,Berlin Office,60020 - Salary and Wages,...,NaN,,,,Germany,,,"Salaried, full-time",,
1,158,,,Carlo Wolf,"Vice-President of Business Development, EMEA",04/01/2023,11/30/2023,Sales,Berlin Office,60020 - Salary and Wages,...,NaN,,,,Germany,,,"Salaried, part-time",,
3,211,,,Robert La Frentz,Account Solutions Architect Manager,06/17/2024,12/15/2024,Sales,"San Jose, CA - HQ",60020 - Salary and Wages,...,NaN,,,California,United States,,,"Salaried, full-time",,
4,172,,,Jason Hilton,Client Executive,08/01/2023,05/15/2024,Sales,"San Jose, CA - HQ",60020 - Salary and Wages,...,NaN,,,California,United States,,,"Salaried, full-time",,
5,70,,,John Min,Client Executive,12/13/2021,04/08/2022,Sales,"San Jose, CA - HQ",60020 - Salary and Wages,...,NaN,,,California,United States,,,"Salaried, full-time",,


In [14]:
# ── Validation Summary ────────────────────────────────────────────────────────
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"Total rows in output          : {len(output_df)}")
print(f"Contractor rows (should be 0) : {(output_df['Employee Type'].str.lower() == 'contractor').sum()}")
print(f"Blank Emp IDs   (should be 0) : {(output_df['Emp ID'] == '').sum()}")
print(f"'Unclassified' Departments    : {(output_df['Department'] == 'Unclassified').sum()}")
print(f"'Unclassified' Locations      : {(output_df['Location'] == 'Unclassified').sum()}")
print(f"EOR employees (60010)         : {(output_df['Base Category Name'] == '60010 - Salary and Wages - EOR').sum()}")
print(f"Annual Base Pay nulls         : {output_df['Annual Base Pay'].isna().sum()}")
print(f"Bonus % set to 0.1            : {(output_df['Bonus %'] == 0.1).sum()}")
print()
print("Sample output (first 5 rows):")
output_df.head(5)

VALIDATION SUMMARY
Total rows in output          : 232
Contractor rows (should be 0) : 0
Blank Emp IDs   (should be 0) : 0
'Unclassified' Departments    : 2
'Unclassified' Locations      : 0
EOR employees (60010)         : 4
Annual Base Pay nulls         : 0
Bonus % set to 0.1            : 11

Sample output (first 5 rows):


,Emp ID,Req ID,Req opening ID,Name (masked),Job Title,Start Date,Termination Date,Department,Location,Base Category Name,...,Grade,Supervisor,Work City,Work State,Work Country,Effective Start Date,Effective End Date,Employee Type,Hire Type,Termination Type
0,130,,,Marcus Oliver Menzel,"Director of Sales, EMEA",11/01/2022,08/12/2023,Sales,Berlin Office,60020 - Salary and Wages,...,NaN,,,,Germany,,,"Salaried, full-time",,
1,158,,,Carlo Wolf,"Vice-President of Business Development, EMEA",04/01/2023,11/30/2023,Sales,Berlin Office,60020 - Salary and Wages,...,NaN,,,,Germany,,,"Salaried, part-time",,
3,211,,,Robert La Frentz,Account Solutions Architect Manager,06/17/2024,12/15/2024,Sales,"San Jose, CA - HQ",60020 - Salary and Wages,...,NaN,,,California,United States,,,"Salaried, full-time",,
4,172,,,Jason Hilton,Client Executive,08/01/2023,05/15/2024,Sales,"San Jose, CA - HQ",60020 - Salary and Wages,...,NaN,,,California,United States,,,"Salaried, full-time",,
5,70,,,John Min,Client Executive,12/13/2021,04/08/2022,Sales,"San Jose, CA - HQ",60020 - Salary and Wages,...,NaN,,,California,United States,,,"Salaried, full-time",,


In [15]:
# ── Export to Excel ───────────────────────────────────────────────────────────
output_df.to_excel(OUTPUT_FILE, index=False)
print(f"✓ Output saved to: {OUTPUT_FILE}")
print(f"  Rows exported : {len(output_df)}")
print(f"  Columns       : {list(output_df.columns)}")

✓ Output saved to: zededa_hris_output_mar27.xlsx
  Rows exported : 232
  Columns       : ['Emp ID', 'Req ID', 'Req opening ID', 'Name (masked)', 'Job Title', 'Start Date', 'Termination Date', 'Department', 'Location', 'Base Category Name', 'Bonus Category Name', 'Annual Base Pay', 'Bonus %', 'Annualized Stock Comp', 'Annualized Sales Comp', 'Grade', 'Supervisor', 'Work City', 'Work State', 'Work Country', 'Effective Start Date', 'Effective End Date', 'Employee Type', 'Hire Type', 'Termination Type']
